In [ ]:
from pathlib import Path
import importlib.util
import sys
import warnings

from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold

from skopt import BayesSearchCV
from skopt.space import Categorical, Integer, Real
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

support_path_candidates = [
    Path("cuda_training_support.py"),
    Path("..") / "cuda_training_support.py",
]
SUPPORT_PATH = next((path.resolve() for path in support_path_candidates if path.exists()), None)
if SUPPORT_PATH is None:
    raise FileNotFoundError("Cannot find cuda_training_support.py")

support_spec = importlib.util.spec_from_file_location("cuda_training_support", SUPPORT_PATH)
if support_spec is None or support_spec.loader is None:
    raise ImportError(f"Cannot load training support module: {SUPPORT_PATH}")

cuda_training_support = importlib.util.module_from_spec(support_spec)
sys.modules["cuda_training_support"] = cuda_training_support
support_spec.loader.exec_module(cuda_training_support)

PROJECT_ROOT = SUPPORT_PATH.parent

current_dir = Path.cwd().resolve()
if (current_dir / "lightgbm").is_dir():
    sys.path = [
        path
        for path in sys.path
        if Path(path or current_dir).resolve() != current_dir
    ]

import lightgbm as lgb

NOTEBOOK_RANDOM_SEED = 114514
NOTEBOOK_TEST_SIZE = None
NOTEBOOK_DEVICE_TYPE = "gpu"
NOTEBOOK_BAYES_N_ITER = 48
NOTEBOOK_CV_FOLDS = 5
NOTEBOOK_MODEL_N_JOBS = cuda_training_support.resolve_recommended_model_n_jobs()
NOTEBOOK_SEARCH_N_JOBS = 1
NOTEBOOK_SMOTE_K_NEIGHBORS = 3
NOTEBOOK_SMOTE_SAMPLING_STRATEGY = 0.75
NOTEBOOK_EARLY_STOPPING_ROUNDS = 300
NOTEBOOK_EARLY_STOPPING_VALIDATION_FRACTION = 0.15
NOTEBOOK_BAYES_SCORING = "roc_auc"
NOTEBOOK_BAYES_VERBOSE = 0
NOTEBOOK_TQDM_DESC = "LightGBM Single Path BayesSearchCV"
NOTEBOOK_INITIAL_THRESHOLD = 0.42
NOTEBOOK_THRESHOLD_SELECTION_METRIC = "F1"
NOTEBOOK_THRESHOLD_PROBE_THRESHOLDS = sorted(
    set(np.round(np.linspace(0.30, 0.70, 41), 3).tolist() + [NOTEBOOK_INITIAL_THRESHOLD])
)
NOTEBOOK_CALIBRATION_METHOD = "isotonic"
NOTEBOOK_MODEL_OUTPUT_PATH = PROJECT_ROOT / "models" / "lightgbm_cuda_model.txt"
NOTEBOOK_PREPROCESSOR_OUTPUT_PATH = PROJECT_ROOT / "models" / "lightgbm_cuda_preprocessor.joblib"
NOTEBOOK_MANIFEST_OUTPUT_PATH = PROJECT_ROOT / "models" / "lightgbm_cuda_inference_assets.json"

NOTEBOOK_LGBM_SEARCH_SPACES = {
    "num_leaves": Integer(24, 127),
    "learning_rate": Real(1e-2, 8e-2, prior="log-uniform"),
    "n_estimators": Integer(1200, 4500),
    "max_depth": Categorical([4, 5, 6, 7, 8, 9]),
    "subsample": Real(0.75, 1.00),
    "colsample_bytree": Real(0.75, 1.00),
    "min_child_samples": Integer(10, 80),
    "min_split_gain": Real(1e-4, 0.12, prior="log-uniform"),
    "reg_alpha": Real(1e-3, 0.6, prior="log-uniform"),
    "reg_lambda": Real(1e-2, 8.0, prior="log-uniform"),
}

# Load external Excel file with train and test sheets
EXTERNAL_DATA_PATH = Path("data-2.xlsx")
if not EXTERNAL_DATA_PATH.exists():
    raise FileNotFoundError(f"Cannot find external data file: {EXTERNAL_DATA_PATH}")

train_df = pd.read_excel(EXTERNAL_DATA_PATH, sheet_name="train")
test_df = pd.read_excel(EXTERNAL_DATA_PATH, sheet_name="test")

# Get compound name column (first column)
compound_name_column = train_df.columns[0]
DEFAULT_TARGET_COLUMN = cuda_training_support.DEFAULT_TARGET_COLUMN

# Store compound names for grouping in cross-validation
train_compound_names = train_df[compound_name_column]

# Extract features and target from train and test sets
X_train = train_df.drop(columns=[DEFAULT_TARGET_COLUMN, compound_name_column])
y_train = train_df[DEFAULT_TARGET_COLUMN]

X_test = test_df.drop(columns=[DEFAULT_TARGET_COLUMN, compound_name_column])
y_test = test_df[DEFAULT_TARGET_COLUMN]

# Create compound group IDs for stratified group cross-validation
unique_compounds = train_compound_names.unique()
compound_to_id = {compound: idx for idx, compound in enumerate(unique_compounds)}
compound_group_ids = train_compound_names.map(compound_to_id).values

# Create a mock prepared dictionary to maintain compatibility with existing code structure
prepared = {
    "X_train": X_train,
    "X_test": X_test,
    "y_train": y_train,
    "y_test": y_test,
    "retention_time_diagnostics": None
}

NOTEBOOK_CONFIG = cuda_training_support.build_notebook_run_config(
    bayes_n_iter=NOTEBOOK_BAYES_N_ITER,
    cv_folds=NOTEBOOK_CV_FOLDS,
    random_seed=NOTEBOOK_RANDOM_SEED,
    test_size=NOTEBOOK_TEST_SIZE,
    device_type=NOTEBOOK_DEVICE_TYPE,
    model_n_jobs=NOTEBOOK_MODEL_N_JOBS,
    search_n_jobs=NOTEBOOK_SEARCH_N_JOBS,
    smote_k_neighbors=NOTEBOOK_SMOTE_K_NEIGHBORS,
    smote_sampling_strategy=NOTEBOOK_SMOTE_SAMPLING_STRATEGY,
    early_stopping_rounds=NOTEBOOK_EARLY_STOPPING_ROUNDS,
    early_stopping_validation_fraction=NOTEBOOK_EARLY_STOPPING_VALIDATION_FRACTION,
    calibration_method=NOTEBOOK_CALIBRATION_METHOD,
    initial_threshold=NOTEBOOK_INITIAL_THRESHOLD,
)

# Use StratifiedGroupKFold to ensure same compound stays together in train/validation splits
NOTEBOOK_CV = StratifiedGroupKFold(
    n_splits=NOTEBOOK_CONFIG.cv_folds,
    shuffle=True,
    random_state=NOTEBOOK_CONFIG.random_seed,
)

DATA_PATH = cuda_training_support.resolve_data_path(start_dir=Path.cwd())

print(
    cuda_training_support.format_notebook_run_summary(
        NOTEBOOK_CONFIG,
        data_path=DATA_PATH,
    )
)
print(
    f"Training config: device_type={NOTEBOOK_CONFIG.device_type} + Intra-fold Borderline-SMOTE + scale_pos_weight after resampling (zero-split candidates auto-rejected)"
)
print(f"Threshold selection metric: {NOTEBOOK_THRESHOLD_SELECTION_METRIC}")
print(f"Number of threshold candidates: {len(NOTEBOOK_THRESHOLD_PROBE_THRESHOLDS)}")
print(f"Model output path: {NOTEBOOK_MODEL_OUTPUT_PATH}")
print(f"Preprocessor output path: {NOTEBOOK_PREPROCESSOR_OUTPUT_PATH}")
print(f"Inference manifest path: {NOTEBOOK_MANIFEST_OUTPUT_PATH}")
print(f"\nData loaded from: {EXTERNAL_DATA_PATH}")
print(f"Compound name column: {compound_name_column}")
print(f"Unique compounds in training set: {len(unique_compounds)}")
print(f"Train set shape (excluding compound name column): {X_train.shape}")
print(f"Test set shape (excluding compound name column): {X_test.shape}")
print(f"Train label distribution:\n{pd.Series(y_train).value_counts()}")
print(f"Test label distribution:\n{pd.Series(y_test).value_counts()}")
print(f"\nCross-validation strategy: StratifiedGroupKFold (same compound never appears in both train and validation splits)")

In [ ]:
# Load data, prepare training data and output pre-training diagnostics
random_seed = NOTEBOOK_CONFIG.random_seed

# Data already loaded from external Excel file
# Combine train and test for retention time analysis only
data = pd.concat([train_df, test_df], axis=0, ignore_index=True)

# Check if RETENTION_TIME column exists (case insensitive check)
retention_time_col = None
for col in data.columns:
    if col.upper() == "RETENTION_TIME":
        retention_time_col = col
        break

retention_time_raw = data[retention_time_col].copy() if retention_time_col is not None else None

# Prepare retention time diagnostics if RETENTION_TIME column exists
retention_time_diagnostics = None
if retention_time_col is not None:
    # Basic retention time diagnostics for external data
    strict_missing = data[retention_time_col].isna().sum()
    cleaned_missing = strict_missing
    recovered_from_text = 0
    multi_value_count = 0
    original_missing = strict_missing
    unparsed_non_missing = 0
    multi_value_examples = []
    
    retention_time_diagnostics = {
        "strict_missing_count": strict_missing,
        "cleaned_missing_count": cleaned_missing,
        "recovered_from_text_count": recovered_from_text,
        "multi_value_count": multi_value_count,
        "original_missing_count": original_missing,
        "unparsed_non_missing_count": unparsed_non_missing,
        "multi_value_examples": multi_value_examples
    }

print("=== Pre-training Diagnostics ===")
print("Total samples:", len(data))
print("Label distribution:")
print(data[DEFAULT_TARGET_COLUMN].value_counts())
print()
if retention_time_raw is not None:
    print("RETENTION_TIME raw values before conversion:")
    print(retention_time_raw.head())
    print()
if retention_time_diagnostics is not None:
    print("RETENTION_TIME cleaning diagnostics:")
    print(f"Strict numeric conversion missing count: {retention_time_diagnostics['strict_missing_count']}")
    print(f"Missing count after cleaning        : {retention_time_diagnostics['cleaned_missing_count']}")
    print(f"Records recovered from text         : {retention_time_diagnostics['recovered_from_text_count']}")
    print(f"Multi-value text records            : {retention_time_diagnostics['multi_value_count']}")
    print(f"Original true missing count         : {retention_time_diagnostics['original_missing_count']}")
    print(f"Unparsed non-missing records        : {retention_time_diagnostics['unparsed_non_missing_count']}")
    print("Multi-value text examples:")
    print(retention_time_diagnostics["multi_value_examples"] or ["<none>"])
    print()
print("Training set shape (original; within folds will first split early stopping, then only apply Borderline-SMOTE augmentation to training subset):", X_train.shape)
print("Test set shape:", X_test.shape)
print("Training set label distribution (original):")
print(pd.Series(y_train).value_counts())
print()
print("Test set label distribution:")
print(pd.Series(y_test).value_counts())
print()
print("Cross-validation grouping info:")
print(f"Number of unique compounds in training set: {len(unique_compounds)}")
print(f"Min samples per compound: {train_compound_names.value_counts().min()}")
print(f"Max samples per compound: {train_compound_names.value_counts().max()}")
print(f"Mean samples per compound: {train_compound_names.value_counts().mean():.2f}")
print()

lgbm_version = cuda_training_support.validate_lightgbm_accelerated_build(
    device_type=NOTEBOOK_CONFIG.device_type,
    random_state=NOTEBOOK_CONFIG.random_seed,
)
print(f"LightGBM {NOTEBOOK_CONFIG.device_type.upper()} preflight:", lgbm_version)

In [ ]:
# Single path training block: Intra-fold Borderline-SMOTE augmentation, then automatically calculate scale_pos_weight based on post-augmentation ratios; zero-split candidates treated as invalid.
metric_order = [
    "AUC",
    "Accuracy",
    "Balanced Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Specificity",
]


def print_metric_block(title, metrics):
    print(title)
    for metric in metric_order:
        print(f"{metric:<18}: {metrics[metric]:.4f}")
    print()


lgb_model = cuda_training_support.build_lgbm_classifier(
    random_state=NOTEBOOK_CONFIG.random_seed,
    device_type=NOTEBOOK_CONFIG.device_type,
    model_n_jobs=NOTEBOOK_CONFIG.model_n_jobs,
    smote_k_neighbors=NOTEBOOK_CONFIG.smote_k_neighbors,
    smote_sampling_strategy=NOTEBOOK_CONFIG.smote_sampling_strategy,
    early_stopping_rounds=NOTEBOOK_CONFIG.early_stopping_rounds,
    early_stopping_validation_fraction=NOTEBOOK_CONFIG.early_stopping_validation_fraction,
)

bayes_search = BayesSearchCV(
    estimator=lgb_model,
    search_spaces=NOTEBOOK_LGBM_SEARCH_SPACES,
    n_iter=NOTEBOOK_CONFIG.bayes_n_iter,
    cv=NOTEBOOK_CV,
    scoring=NOTEBOOK_BAYES_SCORING,
    n_jobs=NOTEBOOK_CONFIG.search_n_jobs,
    verbose=NOTEBOOK_BAYES_VERBOSE,
    random_state=NOTEBOOK_CONFIG.random_seed,
    error_score=0.0,
)

progress_bar = tqdm(
    total=NOTEBOOK_CONFIG.bayes_n_iter,
    desc=NOTEBOOK_TQDM_DESC,
    unit="iter",
)
progress_state = {"completed": 0}


def update_training_progress(_optim_result):
    progress_state["completed"] += 1
    progress_bar.update(1)
    progress_bar.set_postfix(completed=progress_state["completed"], refresh=False)
    return False


try:
    # Fit with compound groups to ensure same compound not split across train/validation
    bayes_search.fit(X_train, y_train, groups=compound_group_ids, callback=update_training_progress)
finally:
    progress_bar.close()

best_lgb = bayes_search.best_estimator_
best_params = dict(bayes_search.best_params_)
best_cv_auc = float(bayes_search.best_score_)
if not np.isfinite(best_cv_auc) or best_cv_auc <= 0.5:
    raise RuntimeError(
        "BayesSearchCV did not find a usable LightGBM model. Current best CV AUC <= 0.5, "
        "indicating that the search space still falls into zero-split/constant prediction regions. "
        "Please continue to relax split constraints and retry."
    )

best_raw_test_proba = best_lgb.predict_proba(X_test)[:, 1]
initial_test_metrics = cuda_training_support.compute_binary_classification_metrics(
    y_true=y_test,
    positive_proba=best_raw_test_proba,
    threshold=NOTEBOOK_CONFIG.initial_threshold,
)

print("Best parameter combination:", best_params)
print("Best cross-validation AUC:", best_cv_auc)
print("Cross-validation strategy: StratifiedGroupKFold (compound-aware)")
print("Training set label distribution before final refit:", getattr(best_lgb, "fit_class_counts_", {}))
print("Intra-fold training subset label distribution:", getattr(best_lgb, "train_split_class_counts_", {}))
print("Intra-fold early stopping validation set label distribution:", getattr(best_lgb, "eval_split_class_counts_", {}))
print("Borderline-SMOTE metadata:", getattr(best_lgb, "resampling_metadata_", {}))
print("scale_pos_weight before resampling:", getattr(best_lgb, "pre_resample_scale_pos_weight_", None))
print("scale_pos_weight after resampling:", getattr(best_lgb, "effective_scale_pos_weight_", None))
print("best_iteration:", getattr(best_lgb, "best_iteration_", None))
print("Training diagnostics:", getattr(best_lgb, "training_diagnostics_", {}))
print(f"The test set metrics below only serve as deployment starting point with initial threshold {NOTEBOOK_CONFIG.initial_threshold:.2f}, not participating in threshold selection.")
print_metric_block(
    f"=== Test set performance (raw probabilities, threshold={NOTEBOOK_CONFIG.initial_threshold:.2f}) ===",
    initial_test_metrics,
)

split_importance_frame = cuda_training_support.build_booster_feature_importance_frame(
    best_lgb.booster_,
    importance_type="split",
    ignore_zero=True,
)
if split_importance_frame.empty:
    print("Skip feature importance plotting: current model has no non-zero split importance. Please check training diagnostics and re-run parameter search.")
else:
    plt.figure(figsize=(10, 6))
    lgb.plot_importance(best_lgb.booster_, max_num_features=20)
    plt.tight_layout()
    plt.show()

In [ ]:
# CV/OOF probability calibration and threshold calibration block: only reference training set OOF, do not look at refit training set scores.
allowed_threshold_metrics = {
    "Accuracy",
    "Balanced Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Specificity",
}
if NOTEBOOK_THRESHOLD_SELECTION_METRIC not in allowed_threshold_metrics:
    raise ValueError(
        f"NOTEBOOK_THRESHOLD_SELECTION_METRIC must be one of {sorted(allowed_threshold_metrics)}"
    )

raw_oof_train_proba = np.zeros(len(y_train), dtype=float)
oof_progress = tqdm(total=NOTEBOOK_CONFIG.cv_folds, desc="OOF Raw Probability CV (Compound-Aware)", unit="fold")

try:
    # Use compound groups to ensure same compound not split across train/validation in OOF
    for fold_idx, (train_idx, valid_idx) in enumerate(NOTEBOOK_CV.split(X_train, y_train, groups=compound_group_ids), start=1):
        fold_model = clone(best_lgb)
        X_fold_train = X_train.iloc[train_idx].reset_index(drop=True)
        y_fold_train = y_train.iloc[train_idx].reset_index(drop=True)
        X_fold_valid = X_train.iloc[valid_idx].reset_index(drop=True)
        
        # Get compound names for this fold to verify no leakage (optional)
        fold_train_compounds = train_compound_names.iloc[train_idx].unique()
        fold_valid_compounds = train_compound_names.iloc[valid_idx].unique()
        overlapping_compounds = set(fold_train_compounds) & set(fold_valid_compounds)
        
        if overlapping_compounds:
            print(f"Warning: Fold {fold_idx} has {len(overlapping_compounds)} overlapping compounds between train and validation!")
        
        fold_model.fit(X_fold_train, y_fold_train)
        raw_oof_train_proba[valid_idx] = fold_model.predict_proba(X_fold_valid)[:, 1]
        oof_progress.update(1)
        oof_progress.set_postfix(fold=fold_idx, refresh=False)
finally:
    oof_progress.close()

probability_calibration_bundle = cuda_training_support.fit_probability_calibrator(
    y_true=y_train,
    positive_proba=raw_oof_train_proba,
    method=NOTEBOOK_CONFIG.calibration_method,
    random_state=NOTEBOOK_CONFIG.random_seed,
)
calibrated_oof_train_proba = probability_calibration_bundle["calibrated_positive_proba"]
best_calibrated_test_proba = cuda_training_support.apply_probability_calibrator(
    best_raw_test_proba,
    probability_calibration_bundle,
)

threshold_probe = cuda_training_support.probe_binary_classification_thresholds(
    y_true=y_train,
    positive_proba=calibrated_oof_train_proba,
    thresholds=NOTEBOOK_THRESHOLD_PROBE_THRESHOLDS,
)

top_f1_thresholds = threshold_probe.sort_values(
    by=["F1", "Balanced Accuracy", "Recall", "threshold"],
    ascending=[False, False, False, True],
).head(10)
top_balanced_thresholds = threshold_probe.sort_values(
    by=["Balanced Accuracy", "F1", "Recall", "threshold"],
    ascending=[False, False, False, True],
).head(10)
selected_threshold_row = cuda_training_support.select_binary_classification_threshold(
    threshold_probe,
    primary_metric=NOTEBOOK_THRESHOLD_SELECTION_METRIC,
    initial_threshold=NOTEBOOK_CONFIG.initial_threshold,
)
SELECTED_THRESHOLD = float(selected_threshold_row["threshold"])

initial_oof_metrics = cuda_training_support.compute_binary_classification_metrics(
    y_true=y_train,
    positive_proba=calibrated_oof_train_proba,
    threshold=NOTEBOOK_CONFIG.initial_threshold,
)
selected_oof_metrics = cuda_training_support.compute_binary_classification_metrics(
    y_true=y_train,
    positive_proba=calibrated_oof_train_proba,
    threshold=SELECTED_THRESHOLD,
)
selected_test_metrics = cuda_training_support.compute_binary_classification_metrics(
    y_true=y_test,
    positive_proba=best_calibrated_test_proba,
    threshold=SELECTED_THRESHOLD,
)
selected_test_confusion_matrix = selected_test_metrics["confusion_matrix"]
threshold_selection_metadata = {
    "selection_source": "cv_oof_calibrated",
    "selection_metric": NOTEBOOK_THRESHOLD_SELECTION_METRIC,
    "initial_threshold": NOTEBOOK_CONFIG.initial_threshold,
    "selected_threshold_row": selected_threshold_row,
    "candidate_threshold_count": len(NOTEBOOK_THRESHOLD_PROBE_THRESHOLDS),
    "oof_metrics_at_selected_threshold": {
        key: value
        for key, value in selected_oof_metrics.items()
        if key != "confusion_matrix"
    },
}

# Note: prepared dictionary may need compound name column handling for inference
# Create a version of prepared without compound name column for compatibility
prepared_for_saving = {
    "X_train": X_train,
    "X_test": X_test, 
    "y_train": y_train,
    "y_test": y_test,
    "retention_time_diagnostics": None
}

saved_artifacts = cuda_training_support.save_lightgbm_inference_artifacts(
    estimator=best_lgb,
    prepared=prepared_for_saving,
    model_path=NOTEBOOK_MODEL_OUTPUT_PATH,
    preprocessor_path=NOTEBOOK_PREPROCESSOR_OUTPUT_PATH,
    manifest_path=NOTEBOOK_MANIFEST_OUTPUT_PATH,
    target_column=DEFAULT_TARGET_COLUMN,
    data_path=DATA_PATH,
    random_seed=NOTEBOOK_CONFIG.random_seed,
    test_size=NOTEBOOK_CONFIG.test_size,
    smote_k_neighbors=NOTEBOOK_CONFIG.smote_k_neighbors,
    smote_sampling_strategy=NOTEBOOK_CONFIG.smote_sampling_strategy,
    scoring=NOTEBOOK_BAYES_SCORING,
    classification_threshold=SELECTED_THRESHOLD,
    probability_calibration_bundle=probability_calibration_bundle,
    threshold_selection_metadata=threshold_selection_metadata,
)

print("=== Probability Calibration Summary ===")
print("Requested calibration method:", probability_calibration_bundle["requested_method"])
print("Actual calibration method used:", probability_calibration_bundle["method"])
if probability_calibration_bundle.get("fallback_reason"):
    print("Calibration fallback reason:", probability_calibration_bundle["fallback_reason"])
print("OOF raw probability metrics:", probability_calibration_bundle["raw_metrics"])
print("OOF calibrated probability metrics:", probability_calibration_bundle["calibrated_metrics"])
print()
print("=== Training Set OOF Threshold Probe: Top 10 by F1 ===")
display(top_f1_thresholds)
print("=== Training Set OOF Threshold Probe: Top 10 by Balanced Accuracy ===")
display(top_balanced_thresholds)
print(f"Selected threshold: {SELECTED_THRESHOLD:.3f}")
print(f"Threshold selection metric: {NOTEBOOK_THRESHOLD_SELECTION_METRIC}")
print(f"Cross-validation strategy: Compound-aware StratifiedGroupKFold")
print_metric_block(
    f"=== Training Set OOF Performance (calibrated probabilities, threshold={NOTEBOOK_CONFIG.initial_threshold:.2f}) ===",
    initial_oof_metrics,
)
print_metric_block("=== Training Set OOF Performance (calibrated probabilities, selected threshold) ===", selected_oof_metrics)
print_metric_block("=== Test Set Performance (calibrated probabilities, selected threshold) ===", selected_test_metrics)
print("Test set confusion matrix (calibrated probabilities, selected threshold):")
print(selected_test_confusion_matrix)
print(f"Model saved to: {saved_artifacts['model_path']}")
print(f"Preprocessor saved to: {saved_artifacts['preprocessor_path']}")
print(f"Inference manifest saved to: {saved_artifacts['manifest_path']}")